In [0]:
# ============================================================
#  NOTEBOOK 01 - RAW → SILVER: VEHÍCULOS
# ============================================================

STORAGE_ACCOUNT = "stdatacolake"
CONTAINER_RAW     = "raw-zone"
CONTAINER_SILVER  = "silver-zone"

RAW_PATH    = f"abfss://{CONTAINER_RAW}@{STORAGE_ACCOUNT}.dfs.core.windows.net/GPS"
SILVER_PATH = f"abfss://{CONTAINER_SILVER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/vehiculos"

In [0]:
# ── LEER CSV DESDE RAW ────────────────────────────────────────
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

schema_vehiculos = StructType([
    StructField("id_vehiculo",   IntegerType(), nullable=False),
    StructField("placa",         StringType(),  nullable=False),
    StructField("tipo",          StringType(),  nullable=True),
    StructField("capacidad_kg",  DoubleType(),  nullable=True),
    StructField("conductor",     StringType(),  nullable=True),
])

df_raw = spark.read \
    .schema(schema_vehiculos) \
    .option("header", "true") \
    .csv(f"{RAW_PATH}/Datos_vehiculos.csv")

print(f"✅ Raw leído: {df_raw.count()} filas")
df_raw.show(5)

In [0]:
# ── LIMPIEZA Y TRANSFORMACIONES ───────────────────────────────
from pyspark.sql.functions import col, trim, upper, current_timestamp

df_silver = df_raw \
    .filter(col("id_vehiculo").isNotNull()) \
    .filter(col("placa").isNotNull()) \
    .withColumn("placa",      upper(trim(col("placa")))) \
    .withColumn("tipo",       trim(col("tipo"))) \
    .withColumn("conductor",  trim(col("conductor"))) \
    .dropDuplicates(["id_vehiculo"]) \
    .withColumn("_updated_at", current_timestamp())

print(f"✅ Silver listo: {df_silver.count()} filas")
df_silver.show(5)

In [0]:
# ── GUARDAR COMO DELTA EN SILVER ─────────────────────────────
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(SILVER_PATH)

print(f"✅ Guardado en Silver: {SILVER_PATH}")